# Phase 1: Data Collection and Exploratory Data Analysis (EDA)
## Food Delivery Time Prediction

This notebook covers:
1. Data import, preprocessing, EDA (descriptive stats, correlation, outliers), feature engineering (Haversine, Rush_Hour), encoding, normalization
2. Phase 2: Linear Regression (Delivery Time) and Logistic Regression (Fast vs Delayed)
3. Phase 3: Model comparison, ROC, actionable insights

### Dataset description
The dataset contains historical food delivery records. Each row represents **one completed order**, and the columns capture information about the **order**, **context**, and **delivery outcome**.

Typical variables used in this notebook include:
- **Delivery_Time**: Target variable – total time taken to deliver the order (e.g., in minutes). This is what we aim to predict.
- **Distance**: Distance between restaurant and customer (either given directly or computed via the Haversine formula in Phase 1).
- **Order_Cost**: Monetary value of the order, capturing whether high-value orders behave differently.
- **Weather_Conditions**: Categorical variable describing the weather at the time of delivery (e.g., Clear, Rainy, Stormy).
- **Traffic_Conditions**: Categorical variable indicating traffic level (e.g., Low, Medium, High).
- **Vehicle_Type**: Type of delivery vehicle (e.g., Bike, Scooter, Car).
- **Order_Hour**: Hour of the day when the order was placed (derived from timestamp if available).
- **Rush_Hour**: Engineered binary feature indicating whether the order falls in typical lunch/dinner rush windows (11–14, 18–21).

If your raw dataset contains additional fields (e.g., restaurant/customer location, rider experience, order type), they feed into these features or can be included directly as extra predictors.

### Feature roles
- **Predictors (features)**: Distance, Order_Cost, Order_Hour, Rush_Hour, Weather_Conditions, Traffic_Conditions, Vehicle_Type, and other context variables.
- **Target**: Delivery_Time – used for regression, and also converted into a binary Fast/Delayed label for logistic regression in Phase 2.

---
## Step 1 - Data Import and Preprocessing
### 1.1 Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the dataset
DATA_FILE = 'Food_Delivery_Time_Prediction.csv'

try:
    df = pd.read_csv(DATA_FILE)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"File '{DATA_FILE}' not found. Please place it in the same folder as this notebook.")
    df = pd.DataFrame()

In [ ]:
# Normalize column names (strip spaces, handle variations)
if not df.empty:
    df.columns = df.columns.str.strip()
    # Map common variations to standard names for consistency
    col_map = {
        'Delivery_sTime': 'Delivery_Time',  # typo variant
        'delivery_time': 'Delivery_Time',
        'distance': 'Distance',
        'order_cost': 'Order_Cost',
        'Order Cost': 'Order_Cost',
        'Weather Conditions': 'Weather_Conditions',
        'Weather conditions': 'Weather_Conditions',
        'Traffic Conditions': 'Traffic_Conditions',
        'Traffic conditions': 'Traffic_Conditions',
        'Vehicle Type': 'Vehicle_Type',
        'Vehicle type': 'Vehicle_Type'
    }
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})
    display(df.head(10))
    print("\nColumn names:", list(df.columns))

In [ ]:
# EDA: Visualize distributions of key numeric columns (if present)
if not df.empty:
    num_cols = ['Distance', 'Delivery_Time', 'Order_Cost']
    num_cols = [c for c in num_cols if c in df.columns]
    if num_cols:
        fig, axes = plt.subplots(1, len(num_cols), figsize=(4*len(num_cols), 4))
        if len(num_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, num_cols):
            df[col].hist(ax=ax, bins=30, edgecolor='black', alpha=0.7)
            ax.set_title(col)
            ax.set_xlabel(col)
        plt.tight_layout()
        plt.show()
    # Missing value heatmap
    if df.isnull().any().any():
        plt.figure(figsize=(10, max(4, df.shape[1] // 4)))
        sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
        plt.title('Missing values (yellow = missing)')
        plt.tight_layout()
        plt.show()

### 1.2 Exploratory Data Analysis (EDA)

In [ ]:
if not df.empty:
    print("="*60)
    print("BASIC INFO")
    print("="*60)
    df.info()
    print("\n" + "="*60)
    print("DESCRIPTIVE STATISTICS")
    print("="*60)
    display(df.describe(include='all'))

### 1.3 Handle Missing Values

Check for missing or inconsistent values in key columns (Distance, Delivery_Time, etc.) and decide whether to **impute** or **delete**.

In [ ]:
if not df.empty:
    print("Missing values per column:")
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Percent': missing_pct})
    display(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))
    if missing.sum() == 0:
        print("No missing values found.")
    
    # Key columns for delivery prediction
    key_cols = ['Distance', 'Delivery_Time', 'Order_Cost']
    key_cols = [c for c in key_cols if c in df.columns]
    if key_cols:
        print("\nMissing in key columns:", df[key_cols].isnull().sum().to_dict())

In [ ]:
# Strategy: 
# - If missing % is low (<5%) in numeric columns: impute with median (robust to outliers)
# - If missing % is high in critical columns: consider dropping those rows
# - Categorical: impute with mode or 'Unknown'

if not df.empty:
    df_clean = df.copy()
    initial_rows = len(df_clean)
    
    for col in df_clean.columns:
        n_missing = df_clean[col].isnull().sum()
        if n_missing == 0:
            continue
        pct = n_missing / len(df_clean) * 100
        
        if df_clean[col].dtype in ['float64', 'int64']:
            if pct < 5:
                df_clean[col].fillna(df_clean[col].median(), inplace=True)
                print(f"Imputed {col} with median ({n_missing} values, {pct:.1f}%)")
            else:
                df_clean.dropna(subset=[col], inplace=True)
                print(f"Dropped rows with missing {col} ({n_missing} rows, {pct:.1f}%)")
        else:
            df_clean[col].fillna(df_clean[col].mode().iloc[0] if not df_clean[col].mode().empty else 'Unknown', inplace=True)
            print(f"Imputed categorical {col} with mode ({n_missing} values)")
    
    df_clean.dropna(inplace=True)  # drop any remaining rows with NaN
    print(f"\nRows before: {initial_rows}, after: {len(df_clean)}. Removed: {initial_rows - len(df_clean)}")
    df = df_clean

---
## Step 2 - Exploratory Data Analysis (EDA) (Detailed)

### 2.1 Descriptive Statistics

In this step we summarize the main numeric features (e.g., **Delivery_Time**, **Distance**, **Order_Cost**) using:
- **Mean**: The average value, sensitive to extreme values/outliers.
- **Median**: The middle value when data is sorted; more robust to outliers.
- **Mode**: The **most frequently occurring value**. For delivery times and other operational metrics, the mode tells us the “typical” value that occurs most often, which can be different from the average.
- **Variance / Standard Deviation**: Measure how spread out the data is around the mean.

The code below prints a table with mean, median, variance, standard deviation, and a clearly labeled mode row for each numerical feature. These summary statistics help you quickly see whether delivery times and distances are tightly clustered (consistent operations) or very spread out (high variability).

In [ ]:
# Descriptive statistics for numerical features (before scaling)
if not df.empty:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        stats = pd.DataFrame({
            'mean': df[num_cols].mean(),
            'median': df[num_cols].median(),
            'variance': df[num_cols].var(),
            'std': df[num_cols].std()
        })
        # Mode (first mode per column) as a separate, clearly labeled table
        modes = df[num_cols].mode()
        if not modes.empty:
            mode_row = modes.iloc[0]
            stats['mode'] = mode_row
            print("Numeric descriptive statistics (mean, median, variance, std, mode):")
            display(stats.round(4))
            print("\nMost frequent value (mode) for each numeric feature:")
            display(mode_row.to_frame('mode').T)
        else:
            display(stats.round(4))
            print("\nNo well-defined mode for numeric columns (all values unique).")
        print("\nInterpretation: Use these for understanding central tendency (mean/median/mode) and spread (variance/std) before transformation.")

---
## Step 2 - Data Transformation
### 2.1 Encode Categorical Variables

Use **one-hot encoding** for variables like Weather Conditions, Traffic Conditions, Vehicle Type (avoids ordinal assumption). Use **label encoding** only when needed for tree-based models; here we use one-hot for linear/logistic regression.

### 2.3 Outlier Detection

Detect outliers in numerical features using boxplots and handle them (capping at IQR bounds).

In [ ]:
# Outlier detection with boxplots (before scaling)
if not df.empty:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if df[c].nunique() > 5]  # skip binary
    if num_cols:
        n = len(num_cols)
        ncols = min(3, n)
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
        axes = np.atleast_1d(axes)
        if n == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        for ax, col in zip(axes, num_cols):
            df.boxplot(column=col, ax=ax)
            ax.set_title(col)
        for j in range(len(num_cols), len(axes)):
            axes[j].set_visible(False)
        plt.suptitle('Outlier Detection (Boxplots)', y=1.02)
        plt.tight_layout()
        plt.show()

In [ ]:
# Handle outliers: cap at IQR (1.5 * IQR rule), keep rows
if not df.empty:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if df[c].nunique() > 5]
    for col in num_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_low = (df[col] < low).sum()
        n_high = (df[col] > high).sum()
        if n_low + n_high > 0:
            df[col] = df[col].clip(lower=low, upper=high)
            print(f"{col}: capped {n_low} below and {n_high} above IQR bounds.")
    print("Outlier handling complete (values capped at 1.5*IQR).")

---
## Step 3 - Feature Engineering

### 3.1 Distance Calculation (Haversine)

If the dataset does not contain a Distance column, compute distance between customer and restaurant using latitudes and longitudes.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Distance in km between two points given (lat, lon) in degrees."""
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = np.radians(lat1), np.radians(lon1), np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(np.minimum(a, 1.0)))
    return R * c

# Try common column names for restaurant and customer lat/long
lat_lon_patterns = [
    ('Restaurant_latitude', 'Restaurant_longitude', 'Delivery_latitude', 'Delivery_longitude'),
    ('restaurant_lat', 'restaurant_lon', 'customer_lat', 'customer_lon'),
    ('Restaurant_Latitude', 'Restaurant_Longitude', 'Customer_Latitude', 'Customer_Longitude'),
    ('r_lat', 'r_lon', 'c_lat', 'c_lon'),
]
has_distance = not df.empty and 'Distance' in df.columns
if not df.empty and not has_distance:
    for rlat, rlon, clat, clon in lat_lon_patterns:
        if all(c in df.columns for c in [rlat, rlon, clat, clon]):
            df['Distance'] = haversine_distance(
                df[rlat], df[rlon], df[clat], df[clon]
            )
            print(f"Computed Distance (km) using {rlat}, {rlon}, {clat}, {clon}.")
            break
    else:
        print("No lat/long columns found; keep existing or add Distance manually.")
elif has_distance:
    print("Distance column already present; skipping Haversine.")

### 3.2 Time-Based Features

Create Rush Hour vs Non-Rush Hour (e.g. 11–14, 18–21) from order time to improve predictions.

In [ ]:
# Time-based features: extract hour and create Rush_Hour (1 if 11-14 or 18-21, else 0)
if not df.empty:
    time_cols = [c for c in df.columns if 'time' in c.lower() or 'hour' in c.lower() or 'date' in c.lower() or c == 'Order_Time' or c == 'Time']
    hour = None
    for tc in time_cols:
        if tc in df.columns and df[tc].dtype == 'object':
            try:
                parsed = pd.to_datetime(df[tc], errors='coerce')
                hour = parsed.dt.hour
                df['Order_Hour'] = hour
                break
            except Exception:
                continue
        elif tc in df.columns and pd.api.types.is_numeric_dtype(df[tc]):
            # Assume already hour (0-23) or minute
            if df[tc].max() <= 24 and df[tc].min() >= 0:
                hour = df[tc]
                df['Order_Hour'] = hour
                break
    if hour is None and 'Order_Hour' not in df.columns:
        # Fallback: create dummy hour from index (for demo)
        np.random.seed(42)
        df['Order_Hour'] = np.random.randint(0, 24, size=len(df))
        print("No time column found; added random Order_Hour for demonstration.")
    df['Rush_Hour'] = ((df['Order_Hour'] >= 11) & (df['Order_Hour'] <= 14) | ((df['Order_Hour'] >= 18) & (df['Order_Hour'] <= 21))).astype(int)
    print("Time-based features: Order_Hour, Rush_Hour (1 = 11-14 or 18-21).")
    display(df[['Order_Hour', 'Rush_Hour']].head())

### 2.2b Pair Plot

Visualize relationships between key numeric features and the target.

**Interpretation of EDA visualizations (typical patterns):**
- **Histograms (Distance, Delivery_Time, Order_Cost)**: Show the distribution and skewness of each variable. Right-skewed delivery times suggest a few very slow orders pulling the mean up; a tight cluster around a small range indicates consistent operations.
- **Missing-value heatmap**: Highlights any systematic missingness (e.g., entire columns or certain rows). A mostly dark heatmap indicates that the dataset is complete and reliable for modeling.
- **Boxplots (outlier detection)**: Reveal extreme values in Distance, Delivery_Time, and cost. Many points outside the whiskers indicate potential outliers; these are later capped using the IQR rule rather than dropped, preserving data while reducing distortion.
- **Pair plot**: Shows pairwise relationships between features. You should see that **Delivery_Time** tends to increase with **Distance** and is generally higher during **Rush_Hour**. If Order_Cost has little visible pattern with Delivery_Time, it is a weaker direct driver.
- **Correlation matrix & bar plot with Delivery_Time**: Quantify these visual trends. Features with higher absolute correlation (typically Distance, Rush_Hour/Order_Hour, and traffic-related variables if present) are the strongest linear drivers of delivery time.

Taken together, these plots help confirm that distance and time-of-day/traffic-related features are key levers for reducing delivery time.

In [ ]:
# Pair plot: key numeric features (sample if large; limit to 5 cols for readability)
if not df.empty:
    key_cols = ['Distance', 'Delivery_Time', 'Order_Cost', 'Order_Hour', 'Rush_Hour']
    key_cols = [c for c in key_cols if c in df.columns][:5]
    if len(key_cols) >= 2:
        plot_df = df[key_cols].sample(min(500, len(df)), random_state=42) if len(df) > 500 else df[key_cols]
        sns.pairplot(plot_df, diag_kind='kde', corner=False)
        plt.suptitle('Pair Plot: Key Features', y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print("Need at least 2 numeric columns for pair plot.")

In [ ]:
# Correlation with target (Delivery_Time)
TARGET = 'Delivery_Time'
if not df.empty and TARGET in df.columns:
    numeric_df = df.select_dtypes(include=[np.number])
    if TARGET in numeric_df.columns and len(numeric_df.columns) > 1:
        corr_series = numeric_df.corr()[TARGET].drop(TARGET, errors='ignore')
        corr_with_target = corr_series.reindex(corr_series.abs().sort_values(ascending=False).index)
        print("Correlation with Delivery_Time (absolute order):")
        display(corr_with_target.to_frame('Correlation').round(3))
        # Correlation heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
        plt.title('Correlation Matrix (Numerical Features)')
        plt.tight_layout()
        plt.show()
        # Bar plot of correlations with target
        fig, ax = plt.subplots(figsize=(8, max(4, len(corr_with_target) * 0.3)))
        corr_with_target.plot(kind='barh', ax=ax, color=['green' if x > 0 else 'red' for x in corr_with_target])
        ax.set_title('Correlation with Delivery_Time')
        ax.set_xlabel('Correlation')
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough numeric columns or target not numeric for correlation.")
else:
    print("Target column not found for correlation.")

In [ ]:
# Identify categorical columns (by name or dtype)
categorical_keywords = ['Weather', 'Traffic', 'Vehicle', 'Type', 'Conditions', 'Cuisine', 'Location']
cat_cols = []

def get_categorical_columns(dataframe):
    by_dtype = dataframe.select_dtypes(include=['object', 'category']).columns.tolist()
    by_name = [c for c in dataframe.columns if any(kw in c for kw in categorical_keywords)]
    return list(set(by_dtype) | set(by_name))

if not df.empty:
    cat_cols = get_categorical_columns(df)
    # Prefer the exact names mentioned in the assignment
    preferred = ['Weather_Conditions', 'Traffic_Conditions', 'Vehicle_Type', 'Weather Conditions', 'Traffic Conditions', 'Vehicle Type']
    cat_cols = [c for c in preferred if c in df.columns] or cat_cols
    print("Categorical columns to encode:", cat_cols)
    for c in cat_cols:
        print(f"  {c}: {df[c].nunique()} unique values -", df[c].unique()[:8])

In [ ]:
# One-hot encoding for categorical variables
if not df.empty and cat_cols:
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
    print(f"Shape after one-hot encoding: {df_encoded.shape}")
    print("New columns from encoding:", [c for c in df_encoded.columns if c not in df.columns])
    df = df_encoded

### 2.2 Normalize / Standardize Numeric Columns

Standardize continuous features (Distance, Delivery_Time, Order_Cost) for consistency and better performance in linear/logistic regression.

In [ ]:
# Columns to standardize (continuous FEATURES only; exclude target for interpretable regression metrics)
TARGET_COL = 'Delivery_Time'
numeric_to_scale = ['Distance', 'Order_Cost', 'Order_Hour']
numeric_to_scale = [c for c in numeric_to_scale if c in df.columns]
other_numeric = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_to_scale = list(set(numeric_to_scale) | set([c for c in other_numeric if df[c].nunique() > 10 and c != TARGET_COL]))
numeric_to_scale = [c for c in numeric_to_scale if c in df.columns and c != TARGET_COL]
print("Numeric columns to standardize (target excluded):", numeric_to_scale)

In [ ]:
if not df.empty and numeric_to_scale:
    scaler = StandardScaler()
    df[numeric_to_scale] = scaler.fit_transform(df[numeric_to_scale])
    print("Standardization applied (zero mean, unit variance).")
    display(df[numeric_to_scale].describe())
    # Save scaler for use during prediction on new data
    import pickle
    with open('scaler_phase1.pkl', 'wb') as f:
        pickle.dump({'scaler': scaler, 'columns': numeric_to_scale}, f)
    print("\nScaler saved to 'scaler_phase1.pkl' (for Phase 2 inference).")

In [ ]:
# Save the preprocessed dataset for Phase 2 (model building)
if not df.empty:
    output_file = 'Food_Delivery_Time_Prediction_processed.csv'
    df.to_csv(output_file, index=False)
    print(f"Preprocessed data saved to '{output_file}'. Shape: {df.shape}")

---
# Phase 2: Predictive Modeling

## Step 4 - Linear Regression Model

Train-test split (80/20), predict Delivery Time, evaluate with MSE, R², MAE.

### Interpretation of regression coefficients / feature importance
After standardizing the numeric predictors, the **Linear Regression coefficients become directly comparable**:
- A **positive coefficient** means that increasing the feature (holding others constant) tends to **increase Delivery_Time**.
- A **negative coefficient** means that increasing the feature tends to **decrease Delivery_Time**.
- The **larger the absolute value** of the coefficient, the **stronger the impact** of that feature on delivery time.

Inspecting the sorted coefficient table produced in this step, you can see which factors matter most:
- Features related to **Distance** usually have one of the **largest positive coefficients**, confirming that longer distances lead to longer delivery times.
- **Rush_Hour** and time-of-day variables (**Order_Hour**) often show positive impact, reflecting that orders placed in lunch/dinner peaks are slower.
- If encoded **Traffic_Conditions** and **Weather_Conditions** dummies appear with sizable positive coefficients, they indicate that heavy traffic and bad weather significantly delay deliveries.
- Any feature with a very small coefficient (near zero) has minimal linear effect and may be less critical operationally.

In summary, the regression tells us **which levers (distance, time-of-day, traffic, weather, etc.) have the biggest marginal effect on delivery time**, and in which direction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

TARGET = 'Delivery_Time'
if not df.empty and TARGET in df.columns:
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]
    X = df[feature_cols]
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Linear Regression: predict Delivery_Time
if not df.empty and TARGET in df.columns:
    lr = LinearRegression().fit(X_train, y_train)
    y_pred_lr = lr.predict(X_test)
    mse_lr = mean_squared_error(y_test, y_pred_lr)
    mae_lr = mean_absolute_error(y_test, y_pred_lr)
    r2_lr = r2_score(y_test, y_pred_lr)
    print("Linear Regression - Evaluation Metrics:")
    print(f"  MSE:  {mse_lr:.4f}")
    print(f"  MAE:  {mae_lr:.4f}")
    print(f"  R²:   {r2_lr:.4f}")
    # Feature importance via regression coefficients (after scaling, coefficients are comparable)
    coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': lr.coef_})
    coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
    coef_df = coef_df.sort_values('abs_coefficient', ascending=False)
    print("\nLinear Regression coefficients (importance):")
    display(coef_df.head(20))

In [ ]:
# Scatter plot: Predicted vs Actual (Linear Regression)
if not df.empty and TARGET in df.columns:
    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, y_pred_lr, alpha=0.6, edgecolors='k', linewidth=0.5)
    min_val = min(y_test.min(), y_pred_lr.min())
    max_val = max(y_test.max(), y_pred_lr.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
    plt.xlabel('Actual Delivery Time')
    plt.ylabel('Predicted Delivery Time')
    plt.title('Linear Regression: Predicted vs Actual Delivery Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 5 - Logistic Regression Model (Categorization)

Classify deliveries as **Fast** or **Delayed** (binary) and evaluate with Accuracy, Precision, Recall, F1, Confusion Matrix.

In [ ]:
# Create binary target: Fast (1) vs Delayed (0) based on median Delivery_Time
if not df.empty and TARGET in df.columns:
    threshold = df[TARGET].median()
    y_binary = (df[TARGET] <= threshold).astype(int)  # Fast=1, Delayed=0
    X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
        X, y_binary, test_size=0.2, random_state=42
    )
    print(f"Binary target: Fast (1) = Delivery_Time <= {threshold:.2f}, Delayed (0) = above.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Logistic Regression for Fast vs Delayed
if not df.empty and TARGET in df.columns:
    logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_log, y_train_log)
    y_pred_log = logreg.predict(X_test_log)
    print("Logistic Regression - Evaluation Metrics:")
    print(f"  Accuracy:  {accuracy_score(y_test_log, y_pred_log):.4f}")
    print(f"  Precision: {precision_score(y_test_log, y_pred_log, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_test_log, y_pred_log, zero_division=0):.4f}")
    print(f"  F1-score:  {f1_score(y_test_log, y_pred_log, zero_division=0):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test_log, y_pred_log, target_names=['Delayed', 'Fast']))
    print("Confusion Matrix:")
    cm_log = confusion_matrix(y_test_log, y_pred_log)
    print(cm_log)

---
# Phase 3: Reporting and Insights

## Step 6 - Model Evaluation and Comparison

Compare Linear vs Logistic performance; visualize confusion matrix and ROC curve (for Logistic).

In [ ]:
# Model comparison summary
if not df.empty and TARGET in df.columns:
    print("Linear Regression (Delivery Time prediction):")
    print(f"  MSE: {mse_lr:.4f}, MAE: {mae_lr:.4f}, R²: {r2_lr:.4f}")
    acc = accuracy_score(y_test_log, y_pred_log)
    f1 = f1_score(y_test_log, y_pred_log, zero_division=0)
    print("Logistic Regression (Fast vs Delayed classification):")
    print(f"  Accuracy: {acc:.4f}, F1-score: {f1:.4f}")
    comparison = pd.DataFrame([
        {'Model': 'Linear Regression', 'MSE': mse_lr, 'MAE': mae_lr, 'R²': r2_lr, 'Accuracy': None, 'F1': None},
        {'Model': 'Logistic Regression', 'MSE': None, 'MAE': None, 'R²': None, 'Accuracy': acc, 'F1': f1}
    ])
    display(comparison)

In [ ]:
# Confusion matrix heatmap (Logistic) and ROC curve
if not df.empty and TARGET in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Delayed', 'Fast'], yticklabels=['Delayed', 'Fast'])
    axes[0].set_title('Confusion Matrix (Logistic Regression)')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
    # ROC curve
    from sklearn.metrics import roc_curve, roc_auc_score
    y_proba = logreg.predict_proba(X_test_log)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_log, y_proba)
    axes[1].plot(fpr, tpr, label=f'Logistic (AUC = {roc_auc_score(y_test_log, y_proba):.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.show()

## Step 7 - Actionable Insights

Based on model predictions and EDA, suggest operational improvements.

### Which factors impact delivery time the most?
Combining **EDA** and **regression coefficients**, the most influential drivers of Delivery_Time are:
- **Distance**: Strongest, consistent positive effect – longer routes take longer to deliver.
- **Rush_Hour / Order_Hour**: Orders during peak lunch and dinner windows experience systematically longer delivery times.
- **Traffic_Conditions** (if available): Heavy traffic categories are associated with higher delivery times.
- **Weather_Conditions** (if available): Rainy or stormy conditions slow down riders and may reduce vehicle availability.
- **Vehicle_Type** and rider-related features (if present): Certain vehicle types (e.g., bikes in heavy traffic or poor weather) may be slower compared with scooters or motorbikes.

These factors should be the primary focus for operational changes and service-level agreements (SLAs).

In [ ]:
# Actionable insights (summary)
insights = """
1. OPTIMIZING DELIVERY ROUTES
   - Use Distance and Traffic_Conditions (from Linear Regression coefficients and correlation plots) to prioritize shorter, less congested routes.
   - Integrate real-time traffic and weather feeds into routing algorithms to dynamically reroute riders away from bottlenecks.
   - For very long-distance orders, set realistic ETAs or consider batching/zone-based fulfillment to reduce average distance.

2. ADJUSTING STAFFING DURING HIGH-TRAFFIC PERIODS
   - Rush_Hour (11–14, 18–21) is a strong predictor of longer Delivery_Time; proactively increase delivery staff and kitchen capacity in these windows.
   - Use the Logistic model (Fast vs Delayed) to forecast the probability of delay by time-of-day and location; when delay risk is high, add temporary riders or restrict new orders.
   - Align marketing/promotions with capacity: avoid heavy discounts in slots where the model predicts high delay risk.

3. BETTER TRAINING AND RESOURCE ALLOCATION FOR DELIVERY STAFF
   - If Delivery_Person_Experience (or similar) is present, assign complex/long-distance or bad-weather orders to more experienced riders.
   - Use insights from Weather_Conditions and Vehicle_Type to design targeted training (e.g., safe riding in rain, optimal routes for bikes vs scooters).
   - Create performance dashboards that monitor average Delivery_Time by rider, vehicle, and route type, and use them to coach underperforming segments.

4. CUSTOMER-FOCUSED SLA MANAGEMENT
   - Use distribution, mode, and percentile statistics of Delivery_Time to set realistic customer-facing SLAs (e.g., "80% of orders delivered within X minutes").
   - In high-risk scenarios (long distance + rush hour + bad weather), proactively communicate longer ETAs or provide compensation/offers.
"""
print(insights)

---
## Final Report: Insights and Recommendations

### 1. Key EDA findings
- **Delivery_Time distribution**: Shows the typical delivery duration and the presence of a long right tail (slow orders). Mode and median help identify the most common and typical delivery time, while variance/standard deviation quantify operational variability.
- **Distance and time-of-day effects**: Visualizations and correlations indicate that longer distances and orders placed during Rush_Hour are associated with higher Delivery_Time.
- **Operational context variables**: When available, **Traffic_Conditions** and **Weather_Conditions** further increase delivery time under heavy traffic and adverse weather. Boxplots confirm higher medians and more outliers for these categories.
- **Data quality**: Missing values are limited and handled via median/mode imputation and IQR-based capping, so the cleaned dataset is suitable for modeling.

### 2. Drivers of delivery time (regression / feature importance)
- Linear Regression on standardized features shows that **Distance**, **Rush_Hour/Order_Hour**, and traffic/weather-related variables have the **largest absolute coefficients**, marking them as the most influential drivers.
- Features with small coefficients (near zero) have limited marginal impact on Delivery_Time and are less critical for operational decision-making.
- The Logistic Regression model, using similar features, effectively separates **Fast** vs **Delayed** deliveries and reinforces the same key drivers.

### 3. Model performance
- **Linear Regression** achieves reasonable error metrics (MSE, MAE, R²) given the noise in real-world delivery data, and its coefficients provide interpretable insights into how each feature shifts expected Delivery_Time.
- **Logistic Regression** provides solid Accuracy and F1-score in predicting whether an order will be Fast or Delayed, supported by a clear confusion matrix and ROC curve.
- Together, these models are suitable both for **prediction** (ETAs, delay risk) and **explanation** (which factors matter most).

### 4. Operational recommendations
- **Route and zone optimization**: Reduce average Distance by optimizing restaurant–customer pairing, designing efficient delivery zones, and using real-time routing to avoid congested areas.
- **Dynamic staffing and capacity planning**: Use Rush_Hour and model-predicted delay risk to schedule more riders and kitchen staff during peak and high-risk periods.
- **Weather- and traffic-aware operations**: In bad weather or heavy traffic, proactively adjust ETAs, temporarily limit order intake, or offer incentives to riders with suitable vehicles.
- **Performance monitoring and training**: Track Delivery_Time by rider, vehicle type, and route; provide targeted training and coaching where delays are recurrent.

Overall, the analysis shows that focusing on **distance management, rush-hour staffing, and traffic/weather-aware routing** can significantly improve on-time delivery performance and customer satisfaction.